# Nuisance factors

**STA 509 2.0 Introduction to Design and Analysis of Experiments** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta509.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Nuisance variation and blocking](#theory-01)
- [Complete-block arrangement](#theory-02)
- [Additive fixed-effects model](#theory-03)
- [ANOVA formulas](#theory-04)
- [Fitted values and diagnostics](#theory-05)
- [Comparisons, missing values and limitations](#theory-06)
- [Incomplete blocks](#theory-07)
- [Latin squares from the course outline](#theory-08)
- [Choosing and interpreting the design](#theory-09)

## Detailed theory · Blocking, RCBD and designs with further control

<a id="theory-01"></a>

### Nuisance variation and blocking
A nuisance factor changes the response but is not the main treatment question. Form blocks of similar units before treatment assignment, then randomize treatments within each block. Examples include field fertility regions, production batches or participants in a complete matched design. Treatment must not be confounded with block.

<a id="theory-02"></a>

### Complete-block arrangement
With a treatments and b complete blocks, every block contains every treatment once in the basic RCBD. A block is a restriction on randomization, not another independent treatment experiment. Useful blocking can remove nuisance variation, though spending error df on unhelpful blocks can reduce efficiency.

<a id="theory-03"></a>

### Additive fixed-effects model
$$Y_{ij}=\mu+\tau_i+\beta_j+\epsilon_{ij}.$$
Apply identifiability constraints or equivalent reference coding. Classical inference assumes independent mean-zero normal errors of common variance and an additive treatment/block mean structure. With one observation per cell, treatment-by-block interaction cannot be estimated separately from error.

<a id="theory-04"></a>

### ANOVA formulas
Let $T_i$ be treatment totals, $B_j$ block totals, $G$ the grand total and $CF=G^2/(ab)$:
$$SS_{Trt}=\sum_iT_i^2/b-CF,\quad
SS_{Block}=\sum_jB_j^2/a-CF,$$
$$SS_E=\sum_{ij}y_{ij}^2-CF-SS_{Trt}-SS_{Block}.$$
Df are a−1,b−1,(a−1)(b−1),ab−1. Treatment F is $MS_{Trt}/MS_E$ with its corresponding df. A block F has a different numerator df and interpretation; blocking was chosen for control, not because its post hoc p-value happened to be small.

<a id="theory-05"></a>

### Fitted values and diagnostics
$$\hat y_{ij}=\bar y_{i.}+\bar y_{.j}-\bar y_{..},\quad
e_{ij}=y_{ij}-\hat y_{ij}.$$
Check normal-reference plots, residuals versus fitted values and run order, and treatment/block patterns. The observed data grid, not a vague description of field layout, determines the calculation.

<a id="theory-06"></a>

### Comparisons, missing values and limitations
For a balanced RCBD treatment difference, $SE(\bar y_i-\bar y_k)=\sqrt{2MS_E/b}$ under the additive model. Use appropriate comparison adjustments. Missing cells destroy the simple orthogonal hand formulas; use an appropriate fitted model and adjusted comparisons, not zero filling or an unchanged balanced ANOVA table.

<a id="theory-07"></a>

### Incomplete blocks
When a block cannot hold every treatment, an incomplete-block design may be needed. Treatments and blocks are no longer orthogonal in the usual simple way, so treatment comparisons must adjust for block and the design must be connected for all intended contrasts. A smaller block is not automatically a valid incomplete-block experiment; allocation and estimability must be planned.

<a id="theory-08"></a>

### Latin squares from the course outline
A Latin square controls two blocking directions with a treatments in an a×a array, each treatment appearing once per row and column:
$$Y_{ij}=\mu+\rho_i+\kappa_j+\tau_{t(i,j)}+\epsilon_{ij}.$$
Randomize a valid layout through appropriate row, column and treatment relabelling. Row, column and treatment df are each a−1; error df is (a−1)(a−2). Corresponding SS use marginal-total squares divided by a, minus CF; error is the remaining total SS. The additive model excludes interactions that this minimal design cannot independently estimate.

<a id="theory-09"></a>

### Choosing and interpreting the design
Use CRD for suitable homogeneous units, RCBD for one helpful complete blocking factor, and a planned Latin square for two relevant blocking directions. The outline names further designs without detailed supplied lecture notes; the explanations here supply foundational understanding rather than reproducing nonexistent additional handouts. Report randomization restrictions, experimental units, replication, model, diagnostics and the population of units to which conclusions apply.

## Concepts covered

**Course points:** Nuisance factors; randomized complete block design; additive model; treatment/block sums of squares; rice-variety trial.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · RCBD: data grid, treatment/block totals and ANOVA

### Question and assumptions

Three treatments are randomized once within each of four blocks. Rows are treatments; columns are blocks. Use an additive treatment-plus-block model.

### Procedure

1. Show the treatment-by-block data grid and row/column totals.
2. Calculate treatment SS and block SS from marginal means.
3. Subtract both from total SS to get error SS.
4. Use treatment df=a−1, block df=b−1 and error df=(a−1)(b−1).

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Treatment | Block 1 | Block 2 | Block 3 | Block 4 | Total | Mean |
| --- | --- | --- | --- | --- | --- | --- |
| A | 8 | 10 | 11 | 13 | 42 | 10.5 |
| B | 9 | 12 | 12 | 15 | 48 | 12 |
| C | 12 | 13 | 15 | 16 | 56 | 14 |
| Block total | 29 | 35 | 38 | 44 | 146 | 12.1667 |

| Source | SS | df | MS | F |
| --- | --- | --- | --- | --- |
| Treatment | 24.6667 | 2 | 12.3333 | 37 |
| Block | 39 | 3 | 13 | 39 |
| Error | 2 | 6 | 0.333333 |  |
| Total | 65.6667 | 11 |  |  |

| Treatment decision quantity | Value |
| --- | --- |
| F | 37 |
| F(.95;2,6) cutoff | 5.14325 |
| Decision | Reject equal treatment means |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


Y=[[8,10,11,13],[9,12,12,15],[12,13,15,16]]; a=len(Y); b=len(Y[0]); N=a*b
row_totals=list(map(sum,Y)); col_totals=[sum(row[j] for row in Y) for j in range(b)]; grand=sum(row_totals)/N
show_table(["Treatment","Block 1","Block 2","Block 3","Block 4","Total","Mean"],[[name,*row,sum(row),mean(row)] for name,row in zip(["A","B","C"],Y)]+[["Block total",*col_totals,sum(row_totals),grand]])
SS_total=sum((v-grand)**2 for row in Y for v in row)
SS_tr=b*sum((mean(row)-grand)**2 for row in Y)
SS_block=a*sum((t/a-grand)**2 for t in col_totals)
SS_error=SS_total-SS_tr-SS_block; dfe=(a-1)*(b-1); MSE=SS_error/dfe
show_table(["Source","SS","df","MS","F"],[
["Treatment",SS_tr,a-1,SS_tr/(a-1),(SS_tr/(a-1))/MSE],
["Block",SS_block,b-1,SS_block/(b-1),(SS_block/(b-1))/MSE],
["Error",SS_error,dfe,MSE,""],["Total",SS_total,N-1,"",""]])
show_table(["Treatment decision quantity","Value"],[["F",SS_tr/(a-1)/MSE],["F(.95;2,6) cutoff",5.143252849784718],["Decision","Reject equal treatment means" if SS_tr/(a-1)/MSE>5.143252849784718 else "Do not reject"]])
assert abs(SS_total-SS_tr-SS_block-SS_error)<1e-12 and (a-1)+(b-1)+dfe==N-1 and MSE>0


### Decision and interpretation

Blocks absorb a known nuisance source. With one observation in each treatment-block cell, interaction cannot be separately estimated in this additive model; a strong treatment-by-block interaction undermines the error interpretation.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
t,b=4,5; tr=np.array([0,2,4,7]); block=np.array([-5,-2,0,3,5]); y=tr[:,None]+block[None,:]+rng.normal(0,2,(t,b)); grand=y.mean()
SST=np.sum((y-grand)**2); ST=b*np.sum((y.mean(1)-grand)**2); SB=t*np.sum((y.mean(0)-grand)**2); SE=SST-ST-SB
Ft=(ST/(t-1))/(SE/((t-1)*(b-1))); Fb=(SB/(b-1))/(SE/((t-1)*(b-1)))
print('yield matrix varieties x blocks\n',np.round(y,1)); print('treatment F',round(Ft,2),'block F',round(Fb,2))
for j in range(b): plt.plot(range(1,t+1),y[:,j],marker='o',label='block '+str(j+1))
plt.xlabel('variety'); plt.ylabel('yield'); plt.legend(ncol=2); plt.show()

### Practice set · RCBD design board
1. A field has a fertility gradient. Propose blocks and describe treatment randomization within each.
2. For t=4 varieties and b=5 blocks, give ANOVA degrees of freedom for treatment, blocks, error, and total.
3. If treatment SS=80 (df=3) and error SS=40 (df=12), calculate treatment F.
4. Explain the no-interaction assumption in words.

<details><summary>Answer sketches</summary>
1. Group nearby plots with similar fertility; randomize all varieties once within each block. 2. 3, 4, 12, 19. 3. MS_T≈26.67, MSE≈3.33, F≈8. 4. Treatment differences are approximately consistent across blocks; strong effect variation by block violates additivity.
</details>

### Debrief
Why should units within a block be similar? What would treatment-by-block interaction mean?

Record the formula, assumptions, and one follow-up question.

## Lab reflection

Which assumptions came from the study design? What changed when you adjusted a parameter?